# First experimentation with Python machine learning code


In [11]:
!pip install langchain langchain_community langchain chromadb pypdf tiktoken langchain_text_splitters langchain_openai


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# import libraries
import os
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI
import json
import requests # type: ignore

# Test chatgpt first:

In [3]:
# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("API_KEY") # Loading the API Key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE") # Loading the API Base Url

model_name = "gpt-4o-mini"

# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

# Create a chat completion
completion = client.chat.completions.create(
    model= model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello, how are you. are you alive?"}
    ]
)

# Print the assistant's reply
print(completion.choices[0].message.content)


Hello! I'm just a computer program, so I don't have feelings or consciousness like a living being. But I'm here and ready to help you! How can I assist you today?


# Load PDF

In [4]:
DOC_PATH = "alphabet_10K_2022.pdf"
CHROMA_PATH = "alphabet_db_name"

# load your pdf doc
loader = PyPDFLoader(DOC_PATH)
pages = loader.load()

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # takes all paragraphs from the PDF and breaks them into smaller chunks

# split the doc into smaller chunks i.e. chunk_size=500
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

In [15]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# get OpenAI Embedding model
embeddings = OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])

# embed the chunks as vectors and load them into the database
db_chroma = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PATH)

In [16]:
# this is an example of a user question (query)
query = 'what are the top risks mentioned in the document that will affect the future of alphabet?'

docs_chroma = db_chroma.similarity_search_with_score(query, k=10)

# generate and answer
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

# Generate answer with LLM

In [19]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [20]:
# Prompt template

PROMPT_TEMPLATE = """
Answer the question based only on the following context: {context}

Answer the question based only on the above context: {question}.

Provide a detailed answer.
Don't justify your answers.
Don't give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

In [21]:
# load retrieved context and user query in the prompt template
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)
print(prompt)

Human: 
Answer the question based only on the following context: ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including but not limited to those described below, which could harm our business, reputation, financial condition, and operating results, and affect the trading price of our Class A
and Class C stock.
Risks Specific to our Company
We generate a significant portion of our revenues from advertising. Reduced spending by advertisers, a loss of partners, or new and existing technologies that block ads online and/or affect our ability to customize ads could harm our
business.
We generated more than 80% of total revenues from online advertising in 2022. Many of our advertisers, companies that distribute our products and services, digital publishers, and content providers can terminate their contracts with us at any time. These
partners may not continue to do business with us if we do not create more value
Table of Contents

In [23]:
model = ChatOpenAI(model_name=model_name, openai_api_key=API_KEY, openai_api_base=OPENAI_API_BASE)
response_text = model.invoke(prompt)
print(response_text)

content='The top risks that will affect the future of Alphabet include:\n\n1. **Advertising Revenue Dependence**: A significant portion of revenues is generated from online advertising. Reduced spending by advertisers, loss of partners, or technologies that block ads could adversely impact business.\n\n2. **Contractual Vulnerability**: Many advertisers and partners can terminate contracts at any time, which could harm revenue, especially if greater value is not provided compared to competitors.\n\n3. **Changes in Advertising and Data Privacy Practices**: Changes to advertising policies and data privacy practices, both from the company and other businesses, may affect the effectiveness and availability of advertising services.\n\n4. **Intellectual Property Risks**: There is a risk of losing trademark protection, particularly with the "Google" brand potentially becoming synonymous with searching. Any impairment of intellectual property rights could harm competition and increase costs.\n\

In [24]:
model = ChatOpenAI(openai_api_key=API_KEY)
response_text = model.invoke(prompt)
print(response_text)

content='1. Reduced spending by advertisers, loss of partners, or technologies that block ads online.\n2. Changes in advertising policies and data privacy practices.\n3. Risk associated with trademarks.\n4. Significant impairment of intellectual property rights.\n5. Concerns regarding privacy and data security.\n6. Software bugs, theft, misuse, defects, vulnerabilities, and security breaches.\n7. The dependence on key members of the senior management team and key personnel.\n8. Increased regulatory scrutiny.\n9. Effects of the war in Ukraine on financial results.\n10. Workforce reduction and office space optimization.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 1677, 'total_tokens': 1793, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai